#### 什么是正则化
所谓“正则化”，就是通过引入人为设定的“规则”（则），来“规范”（正）模型的行为，防止它“为所欲为”（过拟合）。

在机器学习中，过拟合的表现是：模型在训练集上表现极好（误差几乎为 0），但在测试集（新数据）上表现很差。
这是因为模型太复杂了，为了追求极致的低误差，不仅学习了数据背后的真实规律（信号），连数据里的随机噪声和特例也当成了规律学进去了。

为了解决过拟合，我们通常有两个选择：
- 增加数据：给模型看更多的题，让它无法死记硬背，只能去学规律。
- 降低模型复杂度：把模型变“笨”一点，限制它的能力。
正则化就是实现第二种选择的核心手段。它的核心思想是在效果差不多的情况下，越简单的模型越可能是真理。

#### 如何正则化
根据对“复杂度”定义的不同，我们引出了两种最著名的正则化方法：

L2 正则化（Ridge Regression / 岭回归）
- 定义：惩罚项是权重的平方和。
- 直觉：它倾向于让模型的权重变得很小，接近于 0，但不会完全等于 0。
- 效果：这就好比告诉模型：“你可以用所有的特征，但每个特征的影响力都不能太大。”这让模型变得**平滑**，不会因为某个特征的微小波动而产生剧烈反应，从而解决了过拟合。
- $L + \frac{\lambda}{2}||w||^2$。

为什么权重会变小但不为 0？
- 几何直观：L2 正则化的约束区域是一个圆形（高维下是超球体）。这个圆形的边界是光滑的，没有尖角。因此，损失函数等高线与它的交点几乎不可能恰好落在坐标轴上，所以权重只会变小，很难精确为 0。
- 梯度下降视角：在更新权重时，L2 正则化会先将权重乘以一个小于 1 的系数（例如 1 - ηλ），这相当于权重衰减。权重越大，衰减得越多；权重越小，衰减得越少。因此，权重会无限接近 0，但不会被强制变为 0。

L1 正则化（Lasso）
- 定义：惩罚项是权重的绝对值之和。
- 直觉：它倾向于让很多权重直接变成0。
- 效果：这就好比告诉模型：“你只能挑最重要的几个特征用，其他的统统扔掉。”这实现了特征选择，让模型变得非常**稀疏**和简单。
- $L + {\lambda}|w|$

为什么会产生稀疏性？
- 几何直观：想象一个二维参数空间，L1 正则化的约束区域是一个菱形（高维下是多面体）。这个菱形有位于坐标轴上的“尖角”。当损失函数的等高线与这个菱形相交时，交点很容易落在这些尖角上，而尖角就意味着某个维度的权重为 0。
- 梯度下降视角：在更新权重时，L1 正则化会以一个固定的步长向 0 的方向“拉”权重。无论权重本身多小，这个拉力都存在，所以小权重很容易被“拉”过零点，最终稳定在 0。

对比：
| 对比维度 | L1 正则化 (Lasso) | L2 正则化 (Ridge) |
| :--- | :--- | :--- |
| 惩罚项 | 权重的绝对值之和 (`λΣ|w|`) | 权重的平方和 (`λΣw²`) |
| 解的性质 | 稀疏解 (部分权重精确为 0) | 稠密解 (权重趋近于 0 但不为 0) |
| 核心能力 | 特征选择 | 防止过拟合 |
| 几何形状 | 菱形 (有尖角) | 圆形 (光滑) |
| 对异常值 | 相对敏感 | 更鲁棒 (稳定) |
| 计算复杂度 | 稍高 (在 0 点不可导) | 较低 (处处可导) |



In [ ]:
# 正则化在pytorch中的实现
# 3.1. 【关键步骤】定义 L1 正则化函数
def l1_regularization(model, lambda_l1):
    """
    计算模型所有参数的 L1 范数（绝对值之和）
    """
    l1_loss = 0.0
    for param in model.parameters():
        # torch.abs(param).sum() 计算单个参数张量的绝对值之和
        l1_loss += torch.abs(param).sum()
    return lambda_l1 * l1_loss

# 3.2. 【关键步骤】定义 L2 正则化函数
def l2_regularization(model, lambda_l2):
    """
    计算模型所有参数的 L2 范数平方和
    """
    l2_loss = 0.0
    for param in model.parameters():
        # torch.square(param).sum() 计算单个参数张量的平方和
        l2_loss += torch.square(param).sum()
    return lambda_l2 * l2_loss
# 4. 在训练循环中使用

# --- 训练步骤开始 ---
optimizer.zero_grad()

# 4.1 计算原始任务的损失 (例如交叉熵损失)
outputs = model(inputs)
task_loss = criterion(outputs, targets)

# 4.2 计算 L1 惩罚项
l1_lambda = 1e-4  # 设置 L1 正则化强度 λ
l1_penalty = l1_regularization(model, l1_lambda)

# 4.3 将两者相加得到总损失
total_loss = task_loss + l1_penalty

# 4.4 反向传播和优化
total_loss.backward()
optimizer.step()

### 暂退法（dropout）

如果说 L1/L2 正则化是通过“限制参数大小”来让模型变简单，那么 Dropout 就是通过“破坏网络结构”来强迫模型变强。

在神经网络的训练过程中，按照一定的概率（比如 50%），随机地让一部分神经元“暂时失效”（输出置为 0），不参与本次的前向传播和反向传播。

形象类比：
想象一个团队在做项目。没有 Dropout：团队里总有一两个“大腿”（特别强的神经元），其他人就产生了依赖心理，反正有大腿带飞，我就不努力了。结果一旦“大腿”请假（测试数据变化），整个团队就垮了。
有 Dropout：老板（Dropout 机制）每天随机把几个人关进小黑屋（置为 0），不让他们干活。这就强迫剩下的每个人都要独当一面，不能依赖特定的队友。最后，整个团队每个人都变得很强，而且团队配合也更鲁棒了。

#### 暂退法的实现思路

Dropout 的操作在训练和测试阶段是完全不同的：
1. 训练阶段（Training）
随机失活：对于每一层的神经元，生成一个随机掩码（Mask）。如果抽到的概率小于设定值（比如p=0.5），就把该神经元的输出变成 0。
缩放（Scaling）：那么平均下来只有一半的神经元在工作。
为了保证输出的期望值不变，我们需要把剩下的神经元输出放大。
通常的做法是除以保留概率 (1−p) 。比如 p=0.5，剩下的神经元输出就要除以 0.5（也就是乘以 2）。
注：PyTorch 的 nn.Dropout 默认会在训练时自动做这个放大操作（Upscale），所以你不需要手动处理。

2. 测试/推理阶段（Testing / Inference）
全员上岗：关闭 Dropout。所有的神经元都参与计算，不再随机丢弃。
不需要缩放：因为训练时已经对保留的神经元进行了放大，测试时直接使用完整的网络输出即可。
注意：在写代码时，一定要调用 model.eval()，这会自动关闭 Dropout 层，否则你的预测结果每次都会不一样。

代码示例：
```python
import torch
from torch import nn

class NetWithDropout(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.layers = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),
            # 【关键点】在这里加入 Dropout
            # p=0.5 表示有 50% 的概率丢弃神经元
            nn.Dropout(0.5), 
            
            nn.Linear(256, 128),
            nn.ReLU(),
            # 第二层 Dropout，通常越靠近输出层，丢弃率可以设大一点
            nn.Dropout(0.5),
            
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        return self.layers(x)

# 使用示例
model = NetWithDropout()

# 训练时
model.train() 
# 此时 Dropout 生效，随机丢弃神经元

# 测试时
model.eval()  
# 此时 Dropout 自动失效，所有神经元参与计算
```

#### 暂退法能够防止过拟合的原因

Dropout 之所以有效，主要归功于以下两个机制：

1. 打破“共适应性”（Co-adaptation）
这是 Dropout 最核心的贡献。在深层网络中，某些神经元可能会过度依赖前一层特定的神经元组合。
例子：神经元 A 发现只要神经元 B 激活，它就可以偷懒。
Dropout 的作用：既然 B 随时可能被“丢弃”，A 就不得不去学习更本质的特征，而不是依赖 B。这迫使每个神经元都学习到鲁棒的特征。

3. 模拟“集成学习”（Ensemble Learning）
集成学习是指训练很多个不同的模型，然后把它们的结果投票平均，通常效果很好。
Dropout 的视角：每次训练迭代（Batch），由于随机丢弃了不同的神经元，我们实际上是在训练一个不同的子网络。
训练了 1000 个 Epoch，就相当于训练了 1000 个结构略有不同的子网络。
在测试时，我们使用完整网络，这相当于把这 1000 个子网络的结果进行了“平均”。这是一种非常高效的集成学习方法。


| 特性 | L1/L2 正则化 | Dropout (暂退法) |
| :--- | :--- | :--- |
| 作用对象 | 权重参数 ( $ w $ ) | 神经元激活值 (输出) |
| 手段 | 限制权重数值的大小（让模型变平滑） | 随机破坏网络结构（让模型变鲁棒） |
| 数学本质 | 修改损失函数（加惩罚项） | 修改网络结构（随机采样） |
| 最佳实践 | 几乎总是使用（尤其是 L2） | 常用于全连接层，卷积层较少用 |

